# BrainDecoding — Google Colab GPU 秒级解包、混合精度训练与验证集可视化评估工作流

本 Notebook 运行在 **Google Colab GPU 实例**（如 L4 / A100 / T4）上：
1. 自动创建并检查 Google Drive 标准目录（`/content/drive/MyDrive/BrainDecoding/{archives,runs,data,models}`）；
2. 从 Google Drive 秒级解包单文件缓存归档 `archives/features_<modality>.tar` 到本地高速盘 `/content/local_cache`；
3. 在本地 `/content/studies_links` 建立与 CPU 预热阶段完全一致的软链接桥（确保缓存签名 100% 命中）；
4. 通过 `--config` 直接读取 `configs/training/*.yaml` 配置文件（支持 **阶段 1: 词级脑编码器**、**阶段 2.1: 词级脑编码器 + 16 层全局上下文 Transformer**、**阶段 2.2: 词级脑编码器 + 16 层局部窗口滚动修订**、**阶段 2.3: 词级脑编码器 + 16 层全历史因果前缀滚动修订**）；
5. 支持通过 `WINDOW_SECONDS`（或修改 YAML 中的 `workflow.window_seconds`）一键切换单字脑信号时间窗长度（如 `0.5s`、`0.75s`、`1.0s`、`1.25s`、`1.5s`、`2.0s`，直接从 50Hz 连续信号缓存动态切窗，无需重新打包缓存）；
6. **默认开箱即用支持「Step 2：量纲对齐递进消融——档位 A + 档位 B 单种子探针与硬门槛判决」（`RUN_TIER_AB_PROBE = True`, `PROBE_SEED = '0'`）**：单次解包缓存并复用已有 Clean 0s Stage-1 特征后，串行训练档位 A（`null_space`）与档位 B（`amortized_timing` = 档位 A + `TemporalGeometryBiasHead` 连续 $\Delta t$ 自校准），先在 **`val` 验证集**盲选胜出档位，再自动完成 **`test` 留出集** `1x` 单遍推理 vs `Pre-Roll M=4` 特异性审计与 **GO / STOP 硬门槛判决**（`1x` 50w Shifted `< 22.0%` 且 50w Gap `> +35.5 pp`），全部报告独立落盘至云盘 `runs/<dataset>/probe_tier_ab_*_comparison_{val,test}.md`，**100% 零覆盖**已有 `m1`/旧 `m2`/主基线目录；
7. **探针达标 `[GO]` 后一键扩跑 3 种子**：只需将 `RUN_TIER_AB_PROBE = False`、选定胜者 `CONFIG_FILE` 并将 `SEED = '0,1,2'`（保持 `EXP_TAG = 'auto'`），即可自动 **0 秒跳过**已跑完的 `seed-000`，仅补训 `seed-001` 与 `seed-002`；若判决为 `[STOP]`，则立即停机直接采用现有 3 种子结果写论文。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、自动创建目录与配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 【Step 2 默认开箱即用】一键串行完成「量纲对齐递进消融 (lambda_delta=0.45, M=2)」档位 A + 档位 B 单种子探针与 GO/STOP 硬门槛判决：
#   - RUN_TIER_AB_PROBE = True 时，直接点击「全部运行」即可一次跑完（内置旧目录参数指纹自检，100% 零覆盖已有 m1/旧 m2/主基线）：
#       1) 档位 A (null_space, M=2)       -> runs/<dataset>/causal_null_space_prefix_<suffix>/prefix_revision/.../seed-000
#       2) 档位 B (amortized_timing, M=2) -> runs/<dataset>/amortized_timing_prefix_<suffix>/prefix_revision/.../seed-000
#   - 硬门槛判据（仅当 VAL 盲选胜者在 TEST 单次前向 1x 下同时满足 50w Shifted < 22.0% 且 50w Gap > +35.5 pp 时晋级 3 种子，否则立即停机）：
#   - 达标 [GO] 后扩跑胜者 3 种子：将 RUN_TIER_AB_PROBE=False，填入看板打印的胜者 CONFIG_FILE 与 EXP_TAG，设 SEED='0,1,2'（自动 0 秒跳过 seed-000，仅补训 seed-001/002）。
RUN_TIER_AB_PROBE = False             # False（默认主线单配置训练）；True：一键连跑 Step 2 档位 A + 档位 B 单种子探针
PROBE_DATASET = 'pallier2025'         # 探针目标数据集：'pallier2025' (法语 MEG) 或 'chineseeeg2_littleprince' (中文 EEG)
PROBE_SEED = '0'                      # Step 2 锁定单种子 '0'（A+B 训练+评估共约 1.5 小时，严控算力）
PROBE_TAG_SUFFIX = 'm2'               # 探针实验标签后缀（若云盘已有不同超参的旧 *_m2，下方会自动升级为 'm2_ld045' 防覆盖！）
RUN_PROBE_TEST_AUDIT = False          # True：在探针 VAL 盲选胜者后，自动跑 seed-000 TEST 审计并输出 [GO]/[STOP] 硬门槛判决
PROBE_GATE_MAX_SHIFTED_50W = 22.0     # Step 2 晋级硬门槛 ①：1x 单次前向 (M_eval=0) 50 词 Shifted < 22.0%
PROBE_GATE_MIN_GAP_50W = 35.5         # Step 2 晋级硬门槛 ②：1x 单次前向 (M_eval=0) 50 词净间距 Clean - Shifted > +35.5 pp
BASELINE_REF_EXP_TAG = '2_lr4e-4_bs256_window_1.0_siamese_debiased_lamda_0.5_history_window'  # 只读对照的现有 3-seed 主基线标签（0 字节写入）
AUTO_UNASSIGN_RUNTIME = True          # True：全部结果刷入云盘后自动释放 Colab GPU 停止扣费；False：跑完保留交互会话

# 单配置模式参数（当 RUN_TIER_AB_PROBE = False 时生效；超参数已针对 L4 GPU 调优）：
#   - 'configs/training/pallier2025_word_encoder.yaml'                              (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'                       (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'                          (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/pallier2025_prefix_revision.yaml'                           (法语 MEG 阶段 2.3: 词级脑编码器 + 16层全历史因果前缀滚动修订 [1..i, i+r])
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_revision.yaml' (法语 MEG 阶段 2.2: 孪生共模抵消 + 跨状态跳变零化 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml' (法语 MEG 阶段 2.3: 旧基线——全历史因果前缀 + 孪生共模抵消)
#   - 'configs/training/siamese_debiased/pallier2025_causal_null_space_prefix_revision.yaml' (法语 MEG 阶段 2.3 档位 A: Pre-Roll [-12,-19,-27,-36]s + 零空间正交 10.0 + 欧氏跨视野 0.45 + M=2)
#   - 'configs/training/siamese_debiased/pallier2025_amortized_timing_prefix_revision.yaml' (法语 MEG 阶段 2.3 档位 B: 档位 A + TemporalGeometryBiasHead eta=0.4 逐视野自校准 + M=2)
#   - 'configs/training/siamese_debiased/pallier2025_siamese_debiased_context.yaml'  (法语 MEG 阶段 2.1: 孪生共模抵消 + 错位零化正则 + 预提取极速训练)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'                              (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'                       (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'                          (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/chineseeeg2_prefix_revision.yaml'                           (中文 EEG 阶段 2.3: 词级脑编码器 + 16层全历史因果前缀滚动修订 [1..i, i+r])
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_revision.yaml' (中文 EEG 阶段 2.2: 孪生共模抵消 + 跨状态跳变零化 + 预提取极速训练)
#   - 'configs/training/siamese_debiased/chineseeeg2_siamese_debiased_prefix_revision.yaml' (中文 EEG 阶段 2.3: 旧基线——全历史因果前缀 + 孪生共模抵消)
#   - 'configs/training/siamese_debiased/chineseeeg2_causal_null_space_prefix_revision.yaml' (中文 EEG 阶段 2.3 档位 A: Pre-Roll [-12,-19,-27,-36]s + 零空间正交 10.0 + 欧氏跨视野 0.45 + M=2)
#   - 'configs/training/siamese_debiased/chineseeeg2_amortized_timing_prefix_revision.yaml' (中文 EEG 阶段 2.3 档位 B: 档位 A + TemporalGeometryBiasHead eta=0.4 逐视野自校准 + M=2)
CONFIG_FILE = 'configs/training/siamese_debiased/pallier2025_siamese_debiased_prefix_revision.yaml'
MODALITY = 'meg'                      # 对应归档标识: 'meg' (pallier2025) 或 'eeg' (chineseeeg2)
EXP_TAG = 'auto'                      # 训练输出独立实验标签（'auto' 按 CONFIG_FILE 自动隔离至 *_m2，绝不覆盖旧 m1 或旧基线；亦支持手动指定自定义名称）
WORD_EXP_TAG = 'lr4e-4_bs256'                     # 阶段 2 (2.1/2.2/2.3) 热启动只读加载哪个实验标签下的阶段 1 (word) 权重（100% 只读，不向该目录写入任何文件）
WINDOW_SECONDS = 1                 # 单字脑信号时间窗长度（秒，如 0.5, 0.75, 1.0, 1.25, 1.5, 2.0；填 None 则沿用 YAML 中的 window_seconds，默认 1.0s）
WINDOW_START_OFFSET_SECONDS = None    # 相对于词起始时刻的窗口起点偏移（秒，默认 0.0；填 None 则沿用 YAML 配置）
# 下方去偏超参默认均设为 None 以 100% 尊重所选 YAML 自身的配方（如需临时覆盖单个超参再填入数值）：
DEBIAS_MODE = None                    # 去偏档位覆盖：'null_space' (档位 A) | 'amortized_timing' (档位 B) | 'legacy_siamese' | None (沿用 YAML)
SHIFT_MODE = None                     # 错位采样模式覆盖：'preroll' (零环绕只读训练尾缓冲) | 'deterministic_circular' | None (沿用 YAML)
PREROLL_SECONDS = None                # Pre-Roll 缓冲秒数覆盖（如 45.0；填 None 沿用 YAML）
ETA_0 = None                          # 主分支扣除权重 eta_0 / eta_train（填 None 沿用 YAML；档位 A 为 0.0，档位 B 为 0.4）
ETA_DELTA = None                      # 旧版回修态差分扣除权重 eta_delta（填 None 沿用 YAML）
LAMBDA_NULL = None                    # 错位零空间正交惩罚系数 lambda_null（填 None 沿用 YAML；档位 A/B 默认 10.0）
LAMBDA_DELTA = None                   # 单一欧氏跨视界稳定系数 lambda_delta（填 None 沿用 YAML；档位 A/B 默认 0.45）
LAMBDA_DISTILL = None                 # 档位 B 时间偏置蒸馏系数 lambda_distill（填 None 沿用 YAML；档位 B 默认 0.5）
LAMBDA_DELTA_NULL = None              # 旧版跨状态词库跳变零化系数 lambda_delta_null（填 None 沿用 YAML；档位 A/B 已彻底移除）
BETA_DELTA_CM = None                  # 旧版逐分支余弦跳变系数 beta_delta_cm（填 None 沿用 YAML）
SHIFTS_PER_STEP = None                # 每步抽取错位分支数 M（填 None 沿用 YAML；档位 A/B 默认 2，第 2 分支 no_grad 直通均值降噪）
SEED = '0,1,2'                        # 单配置模式随机种子（当 RUN_TIER_AB_PROBE=False 时生效，如 '0,1,2'）
SUBSET_ARGS = '--subject all'         # 或 '--num-subjects 2' / '--subjects sub-01,sub-02'
# ========================================================

_active_seed_raw = PROBE_SEED if RUN_TIER_AB_PROBE else SEED
if isinstance(_active_seed_raw, (list, tuple, set)):
    SEED_LIST = [int(s) for s in _active_seed_raw]
else:
    SEED_LIST = [int(s.strip()) for s in str(_active_seed_raw).strip('[]()').replace(' ', ',').split(',') if s.strip()]
SEED_ARG = ','.join(str(s) for s in SEED_LIST)

WINDOW_ARGS = f'--window-seconds {WINDOW_SECONDS}' if WINDOW_SECONDS is not None and str(WINDOW_SECONDS).strip() != '' else ''
if WINDOW_START_OFFSET_SECONDS is not None and str(WINDOW_START_OFFSET_SECONDS).strip() != '':
    WINDOW_ARGS = f'{WINDOW_ARGS} --window-start-offset-seconds {WINDOW_START_OFFSET_SECONDS}'.strip()

def _resolve_subset_glob(_subset_args: str) -> str:
    _txt = ' '.join(str(_subset_args or '').strip().split())
    if not _txt or _txt in ('--subject all', '--subjects all'):
        return 'all-*'
    _toks = _txt.split()
    if '--num-subjects' in _toks:
        _idx = _toks.index('--num-subjects')
        if _idx + 1 < len(_toks) and _toks[_idx + 1].isdigit():
            _n = int(_toks[_idx + 1])
            return 'sub-01' if _n == 1 else f'subset-n{_n:02d}-*'
    return '*'

SUBSET_GLOB = _resolve_subset_glob(SUBSET_ARGS)

import json as _json_init

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

def _resolve_safe_probe_suffix(_ds: str, _raw_suffix: str) -> str:
    """防覆盖保护：若云盘 *_<suffix> 已存在旧超参（如 lambda_delta != 0.45 或 shifts_per_step != 2），自动切换至新后缀。"""
    _suf = str(_raw_suffix or 'm2').strip().strip('_') or 'm2'
    _ds_root = Path(DRIVE_RESULTS_DIR) / _ds
    if not _ds_root.exists():
        return _suf
    _conflict = False
    for _prefix in ('causal_null_space_prefix', 'amortized_timing_prefix'):
        _cand_dir = _ds_root / f'{_prefix}_{_suf}' / 'prefix_revision'
        if not _cand_dir.exists():
            continue
        for _sum_p in _cand_dir.glob(f'{SUBSET_GLOB}/seed-*/training_summary.json'):
            try:
                _sdata = _json_init.loads(_sum_p.read_text(encoding='utf-8'))
                _siam = _sdata.get('siamese_debiased') or {}
                _ld = float(_siam.get('lambda_delta', -1.0))
                _ld_null = float(_siam.get('lambda_delta_null', 0.0))
                _m_step = int(_siam.get('shifts_per_step', 0))
                if abs(_ld - 0.45) > 1e-4 or _ld_null > 1e-6 or _m_step != 2:
                    _conflict = True
                    break
            except Exception:
                pass
    if _conflict:
        _alt = f'{_suf}_ld045'
        print(f'[防覆盖自动保护] 检测到云盘已有不同超参的 *_{_suf}，自动隔离至新后缀 *_{_alt}（旧目录 0 字节改动）！')
        return _alt
    return _suf

ACTIVE_PROBE_SUFFIX = _resolve_safe_probe_suffix(
    'chineseeeg2_littleprince' if 'chinese' in str(PROBE_DATASET).lower() else 'pallier2025',
    PROBE_TAG_SUFFIX,
)

def _resolve_auto_exp_tag(_cfg_path: str, _raw_tag: str) -> str:
    _low = str(_cfg_path).lower()
    _clean_tag = str(_raw_tag or '').strip()
    if (
        _clean_tag.lower() == 'auto'
        or ('amortized_timing' in _low and _clean_tag.startswith('causal_null_space_prefix_'))
        or ('causal_null_space' in _low and _clean_tag.startswith('amortized_timing_prefix_'))
    ):
        if 'amortized_timing' in _low:
            return f'amortized_timing_prefix_{ACTIVE_PROBE_SUFFIX}'
        if 'causal_null_space' in _low:
            return f'causal_null_space_prefix_{ACTIVE_PROBE_SUFFIX}'
        return Path(_cfg_path).stem
    return _clean_tag

def _build_siamese_cli_args(_cfg_path: str) -> str:
    _low = str(_cfg_path).lower()
    if 'word_encoder' in _low or '/main_word' in _low:
        return ''
    _is_rev = 'revision' in _low
    _parts = []
    if DEBIAS_MODE is not None and str(DEBIAS_MODE).strip() != '' and (not RUN_TIER_AB_PROBE):
        _parts.append(f'--debias-mode {DEBIAS_MODE}')
    if SHIFT_MODE is not None and str(SHIFT_MODE).strip() != '':
        _parts.append(f'--shift-mode {SHIFT_MODE}')
    if PREROLL_SECONDS is not None and str(PREROLL_SECONDS).strip() != '':
        _parts.append(f'--preroll-seconds {PREROLL_SECONDS}')
    if ETA_0 is not None and str(ETA_0).strip() != '' and (not RUN_TIER_AB_PROBE):
        _parts.append(f'--eta-0 {ETA_0}')
    if _is_rev and ETA_DELTA is not None and str(ETA_DELTA).strip() != '':
        _parts.append(f'--eta-delta {ETA_DELTA}')
    if LAMBDA_NULL is not None and str(LAMBDA_NULL).strip() != '':
        _parts.append(f'--lambda-null {LAMBDA_NULL}')
    if _is_rev and LAMBDA_DELTA is not None and str(LAMBDA_DELTA).strip() != '':
        _parts.append(f'--lambda-delta {LAMBDA_DELTA}')
    if LAMBDA_DISTILL is not None and str(LAMBDA_DISTILL).strip() != '':
        _parts.append(f'--lambda-distill {LAMBDA_DISTILL}')
    if _is_rev and LAMBDA_DELTA_NULL is not None and str(LAMBDA_DELTA_NULL).strip() != '':
        _parts.append(f'--lambda-delta-null {LAMBDA_DELTA_NULL}')
    if _is_rev and BETA_DELTA_CM is not None and str(BETA_DELTA_CM).strip() != '':
        _parts.append(f'--beta-delta-cm {BETA_DELTA_CM}')
    if SHIFTS_PER_STEP is not None and str(SHIFTS_PER_STEP).strip() != '':
        _parts.append(f'--shifts-per-step {SHIFTS_PER_STEP}')
    return ' '.join(_parts)

if RUN_TIER_AB_PROBE:
    _probe_ds_norm = 'chineseeeg2_littleprince' if 'chinese' in str(PROBE_DATASET).lower() else 'pallier2025'
    _cfg_prefix = 'chineseeeg2' if _probe_ds_norm == 'chineseeeg2_littleprince' else 'pallier2025'
    MODALITY = 'eeg' if _probe_ds_norm == 'chineseeeg2_littleprince' else 'meg'
    TRAIN_JOBS = [
        {
            'label': '档位 A (Causal Null-Space | lambda_null=10, lambda_delta=0.45, M=2)',
            'short_label': 'Tier A (null_space)',
            'tier_key': 'tier_a',
            'dataset': _probe_ds_norm,
            'stage': 'prefix_revision',
            'config_file': f'configs/training/siamese_debiased/{_cfg_prefix}_causal_null_space_prefix_revision.yaml',
            'exp_tag': f'causal_null_space_prefix_{ACTIVE_PROBE_SUFFIX}',
            'siamese_args': _build_siamese_cli_args(f'configs/training/siamese_debiased/{_cfg_prefix}_causal_null_space_prefix_revision.yaml'),
            'eval_control_args': '--control-config configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml',
        },
        {
            'label': '档位 B (Amortized Timing | Tier A + TemporalGeometryBiasHead eta=0.4, M=2)',
            'short_label': 'Tier B (amortized_timing)',
            'tier_key': 'tier_b',
            'dataset': _probe_ds_norm,
            'stage': 'prefix_revision',
            'config_file': f'configs/training/siamese_debiased/{_cfg_prefix}_amortized_timing_prefix_revision.yaml',
            'exp_tag': f'amortized_timing_prefix_{ACTIVE_PROBE_SUFFIX}',
            'siamese_args': _build_siamese_cli_args(f'configs/training/siamese_debiased/{_cfg_prefix}_amortized_timing_prefix_revision.yaml'),
            'eval_control_args': '--control-config configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml',
        },
    ]
    EXP_TAG = TRAIN_JOBS[0]['exp_tag']
    CONFIG_FILE = TRAIN_JOBS[0]['config_file']
else:
    EXP_TAG = _resolve_auto_exp_tag(CONFIG_FILE, EXP_TAG)
    _cfg_low = str(CONFIG_FILE).lower()
    _single_ds = 'chineseeeg2_littleprince' if 'chineseeeg2' in _cfg_low else 'pallier2025'
    _is_word_stage_cfg = 'word_encoder' in _cfg_low or '/main_word' in _cfg_low
    _single_stage = 'word' if _is_word_stage_cfg else ('prefix_revision' if 'prefix_revision' in _cfg_low else ('context' if 'context' in _cfg_low else 'revision'))
    _is_causal_cfg = ('causal_null_space' in _cfg_low) or ('amortized_timing' in _cfg_low) or (str(SHIFT_MODE or '').strip().lower() == 'preroll') or (str(DEBIAS_MODE or '').strip().lower() in ('null_space', 'amortized_timing'))
    SIAMESE_ARGS = _build_siamese_cli_args(CONFIG_FILE)
    EVAL_CONTROL_ARGS = '--control-config configs/controls/common_mode_subtraction/causal_common_mode_subtraction_control.yaml' if (_is_causal_cfg and not _is_word_stage_cfg) else ''
    TRAIN_JOBS = [
        {
            'label': f'单配置训练 ({Path(CONFIG_FILE).stem})',
            'short_label': EXP_TAG or 'default',
            'tier_key': 'single',
            'dataset': _single_ds,
            'stage': _single_stage,
            'config_file': CONFIG_FILE,
            'exp_tag': EXP_TAG,
            'siamese_args': SIAMESE_ARGS,
            'eval_control_args': EVAL_CONTROL_ARGS,
        }
    ]

# 统一通过环境变量驱动所有路径（与 CPU 预热完全对齐）
os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('RUN_TIER_AB_PROBE         =', RUN_TIER_AB_PROBE)
print('RUN_PROBE_TEST_AUDIT      =', RUN_PROBE_TEST_AUDIT)
print('HARD GATE (1x TEST)       =', f'50w Shifted < {PROBE_GATE_MAX_SHIFTED_50W:.1f}% AND 50w Gap > +{PROBE_GATE_MIN_GAP_50W:.1f}%')
print('SEEDS                     =', SEED_LIST, f'(--seed {SEED_ARG})')
print('MODALITY                  =', MODALITY)
print('WINDOW_SECONDS            =', WINDOW_SECONDS if WINDOW_SECONDS is not None else '(from YAML default 1.0s)')
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')
print('QUEUED TRAIN_JOBS         =', len(TRAIN_JOBS))
for _i, _j in enumerate(TRAIN_JOBS, 1):
    print(f"  [{_i}/{len(TRAIN_JOBS)}] {_j['label']} -> runs/{_j['dataset']}/{_j['exp_tag']}/{_j['stage']}/... (cfg={_j['config_file']})")

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 防崩版初始化虚拟环境（锁定系统 PyTorch/CUDA 版本）+ 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 将 Google Drive 上的单一 .tar 缓存包秒级解包回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 启动 GPU 训练（RUN_TIER_AB_PROBE=True 时自动串行完成档位 A 与档位 B 探针训练；已完成的档位/种子自动 0 秒跳过，断线重连自动从 last.ckpt 续训）
import json

def _is_seed_training_completed(_ds: str, _tag: str, _stage: str, _s: int) -> bool:
    _base = (Path(DRIVE_RESULTS_DIR) / _ds / _tag / _stage) if _tag else (Path(DRIVE_RESULTS_DIR) / _ds / _stage)
    if not _base.exists():
        return False
    for _summary_p in _base.glob(f'{SUBSET_GLOB}/seed-{int(_s):03d}/training_summary.json'):
        try:
            _meta = json.loads(_summary_p.read_text(encoding='utf-8'))
            if _meta.get('status') == 'completed' and ((_summary_p.parent / 'best.ckpt').is_file() or (_summary_p.parent / 'best.pt').is_file()):
                return True
        except Exception:
            pass
    return False

for _job_idx, _job in enumerate(TRAIN_JOBS, 1):
    _job_cfg = _job['config_file']
    _job_tag = _job['exp_tag']
    _job_ds = _job['dataset']
    _job_stage = _job['stage']
    _job_siam_args = _job['siamese_args']
    _pending_seeds = [_s for _s in SEED_LIST if not _is_seed_training_completed(_job_ds, _job_tag, _job_stage, _s)]
    print('\n' + '=' * 86)
    print(f"[训练队列 {_job_idx}/{len(TRAIN_JOBS)}] {_job['label']} | exp_tag={_job_tag} | seeds={SEED_LIST}")
    print('=' * 86)
    if not _pending_seeds:
        print(f"[0秒跳过] {_job['label']} 的全部种子 {SEED_LIST} 已完成训练并存在 best.ckpt，无需重训！")
        continue
    _pending_seed_arg = ','.join(str(_s) for _s in _pending_seeds)
    os.environ['BRAINDECODING_EXP_TAG'] = _job_tag
    _tag_cli = f'--exp-tag {_job_tag}' if _job_tag else ''
    _wtag_cli = f'--word-exp-tag {WORD_EXP_TAG}' if (WORD_EXP_TAG and _job_stage != 'word') else ''
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding train \
        --config {_job_cfg} \
        --seed {_pending_seed_arg} \
        --require-cache \
        --resume auto \
        {_tag_cli} \
        {_wtag_cli} \
        {WINDOW_ARGS} \
        {_job_siam_args} \
        {SUBSET_ARGS}

In [ ]:
# 6. 验证集 (val) 盲选决胜 + 留出测试集 (test) 单种子硬门槛判决（独立落盘云盘 probe_tier_ab_*_comparison_{val,test}.md，零覆盖旧目录）
import json
from IPython.display import Image, display

def _is_split_eval_fresh(_ds: str, _tag: str, _stage: str, _s: int, _split: str) -> bool:
    _base = (Path(DRIVE_RESULTS_DIR) / _ds / _tag / _stage) if _tag else (Path(DRIVE_RESULTS_DIR) / _ds / _stage)
    if not _base.exists():
        return False
    for _cms_p in _base.glob(f'{SUBSET_GLOB}/seed-{int(_s):03d}/exploratory/common_mode_subtraction_{_split}.json'):
        _ckpt_p = _cms_p.parent.parent / 'best.ckpt'
        if _cms_p.is_file() and (not _ckpt_p.is_file() or _cms_p.stat().st_mtime >= _ckpt_p.stat().st_mtime):
            return True
    return False

def _collect_cms_rows_for_job(_job: dict, _split: str) -> list[dict]:
    _rows = []
    _j_base = Path(DRIVE_RESULTS_DIR) / _job['dataset'] / _job['exp_tag'] / _job['stage']
    for _s in SEED_LIST:
        _cms_matches = sorted(
            _j_base.glob(f'{SUBSET_GLOB}/seed-{int(_s):03d}/exploratory/common_mode_subtraction_{_split}.json'),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        for _cms_p in _cms_matches[:1]:
            try:
                _cms_d = json.loads(_cms_p.read_text(encoding='utf-8'))
                for _entry in (_cms_d.get('sweep') or []):
                    _eta = float(_entry.get('eta', 0.0))
                    _c_ref = _entry.get('clean_reference') or {}
                    _s_agg = _entry.get('shifted_aggregate') or {}
                    _c50 = float(_c_ref.get('vocab50_macro_top10', 0.0)) * 100.0
                    _s50 = float(_s_agg.get('mean_vocab50_macro_top10', _s_agg.get('shifted_mean_vocab50_macro_top10', 0.0))) * 100.0
                    _g50 = float(_s_agg.get('clean_minus_shifted_vocab50_macro_top10', (_c50 - _s50) / 100.0)) * 100.0
                    _t1_50 = float(_c_ref.get('vocab50_macro_top1', 0.0)) * 100.0
                    _c_full = float(_c_ref.get('full_vocab_word_macro_top10', _c_ref.get('full_vocab_macro_top10', 0.0))) * 100.0
                    _mic_full = float(_c_ref.get('full_vocab_micro_top10', 0.0)) * 100.0
                    _g_full = float(_s_agg.get('clean_minus_shifted_full_vocab_word_macro_top10', _s_agg.get('clean_minus_shifted_full_vocab_macro_top10', 0.0))) * 100.0
                    _prefix = 'Tier A' if 'null_space' in _job['exp_tag'] else ('Tier B' if 'amortized_timing' in _job['exp_tag'] else _job['short_label'][:10])
                    _mode_tag = f'{_prefix} Raw(1x)' if abs(_eta) < 1e-9 else f'{_prefix} EMA({_eta:g},1x)'
                    _rows.append({
                        'seed': int(_s),
                        'label': _mode_tag,
                        'tier_prefix': _prefix,
                        'config_file': _job['config_file'],
                        'exp_tag': _job['exp_tag'],
                        'm_eval': 0,
                        'eta': _eta,
                        'c50': _c50,
                        's50': _s50,
                        'g50': _g50,
                        't1_50': _t1_50,
                        'c_full': _c_full,
                        'mic_full': _mic_full,
                        'g_full': _g_full,
                        'seed_dir': str(_cms_p.parent.parent),
                    })
            except Exception as _exc:
                print(f'[提示] 读取 {_cms_p} 失败: {_exc}')
    return _rows

# --- 6.1 运行验证集 (VAL) 因果评估 ---
for _job_idx, _job in enumerate(TRAIN_JOBS, 1):
    _job_cfg = _job['config_file']
    _job_tag = _job['exp_tag']
    _job_ds = _job['dataset']
    _job_stage = _job['stage']
    _job_eval_ctrl = _job['eval_control_args']
    _pending_val = [_s for _s in SEED_LIST if not _is_split_eval_fresh(_job_ds, _job_tag, _job_stage, _s, 'val')]
    print('\n' + '=' * 86)
    print(f"[验证集评估 {_job_idx}/{len(TRAIN_JOBS)}] {_job['label']} | exp_tag={_job_tag}")
    print('=' * 86)
    if not _pending_val:
        print(f"[0秒跳过] {_job['label']} 的 VAL 评估已是最新，直接读取已落盘 JSON。")
        continue
    _val_seed_arg = ','.join(str(_s) for _s in _pending_val)
    os.environ['BRAINDECODING_EXP_TAG'] = _job_tag
    _tag_cli = f'--exp-tag {_job_tag}' if _job_tag else ''
    _wtag_cli = f'--word-exp-tag {WORD_EXP_TAG}' if (WORD_EXP_TAG and _job_stage != 'word') else ''
    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
        --config {_job_cfg} \
        --seed {_val_seed_arg} \
        --split val \
        --require-cache \
        {_tag_cli} \
        {_wtag_cli} \
        {WINDOW_ARGS} \
        {_job_eval_ctrl} \
        {SUBSET_ARGS}

# --- 6.2 VAL 盲选决胜看板（严格仅依据 VAL 锁定胜出档位，零测试集窥探） ---
_val_winner_exp_tag = TRAIN_JOBS[0]['exp_tag']
_val_winner_cfg = TRAIN_JOBS[0]['config_file']
_val_winner_prefix = 'Tier A'
if len(TRAIN_JOBS) >= 2:
    _probe_val_rows = []
    for _job in TRAIN_JOBS:
        _probe_val_rows.extend(_collect_cms_rows_for_job(_job, 'val'))
    if _probe_val_rows:
        _best_val_row = max(_probe_val_rows, key=lambda r: (r['g50'], r['g_full'], r['c50']))
        _val_winner_exp_tag = _best_val_row['exp_tag']
        _val_winner_cfg = _best_val_row['config_file']
        _val_winner_prefix = _best_val_row['tier_prefix']
        _lines = [
            '=' * 86,
            f"Step 2 Probe Selection (VAL) | {TRAIN_JOBS[0]['dataset']} | Pre-Roll [-10..-40s] + 1x Online",
            '=' * 86,
            f"{'Candidate (Seed)':<24} | {'50wCln':>6} | {'50wShf':>6} | {'50wGap':>7} | {'50wTp1':>6} | {'FulMac':>6} | {'FulMic':>6}",
            '-' * 86,
        ]
        for _r in _probe_val_rows:
            _nm = f"{_r['label']} s{_r['seed']}"
            _lines.append(
                f"{_nm:<24} | {_r['c50']:>5.2f}% | {_r['s50']:>5.2f}% | {_r['g50']:>+6.2f}% | {_r['t1_50']:>5.2f}% | {_r['c_full']:>5.2f}% | {_r['mic_full']:>5.2f}%"
            )
        _lines.append('-' * 86)
        _lines.append(
            f" * VAL Blind Winner : {_best_val_row['label']} (VAL 50w Gap={_best_val_row['g50']:+.2f}%, FulMic={_best_val_row['mic_full']:.2f}%)"
        )
        _lines.append(
            f" * Winner YAML      : {Path(_val_winner_cfg).name} (exp_tag={_val_winner_exp_tag})"
        )
        _lines.append('=' * 86)
        _board_val_txt = '\n'.join(_lines)
        print('\n' + _board_val_txt)
        _cmp_val_out = Path(DRIVE_RESULTS_DIR) / TRAIN_JOBS[0]['dataset'] / f'probe_tier_ab_{ACTIVE_PROBE_SUFFIX}_comparison_val.md'
        _cmp_val_out.write_text('```text\n' + _board_val_txt + '\n```\n', encoding='utf-8')
        print(f'[已永久保存 VAL 盲选看板至云盘] {_cmp_val_out}')

# --- 6.3 留出测试集 (TEST) 单种子硬门槛审计（1x 单遍推理 vs Pre-Roll M=4 + 自动 [GO]/[STOP] 判决） ---
if RUN_TIER_AB_PROBE and RUN_PROBE_TEST_AUDIT:
    for _job_idx, _job in enumerate(TRAIN_JOBS, 1):
        _job_cfg = _job['config_file']
        _job_tag = _job['exp_tag']
        _job_ds = _job['dataset']
        _job_stage = _job['stage']
        _job_eval_ctrl = _job['eval_control_args']
        if _job_stage == 'word':
            continue
        _pending_test = [_s for _s in SEED_LIST if not _is_split_eval_fresh(_job_ds, _job_tag, _job_stage, _s, 'test')]
        print('\n' + '=' * 86)
        print(f"[TEST 硬门槛审计 {_job_idx}/{len(TRAIN_JOBS)}] {_job['label']} | exp_tag={_job_tag}")
        print('=' * 86)
        if _pending_test:
            _test_seed_arg = ','.join(str(_s) for _s in _pending_test)
            os.environ['BRAINDECODING_EXP_TAG'] = _job_tag
            _tag_cli = f'--exp-tag {_job_tag}' if _job_tag else ''
            _wtag_cli = f'--word-exp-tag {WORD_EXP_TAG}' if WORD_EXP_TAG else ''
            !cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
                --config {_job_cfg} \
                --seed {_test_seed_arg} \
                --split test \
                --require-cache \
                {_tag_cli} \
                {_wtag_cli} \
                {WINDOW_ARGS} \
                {_job_eval_ctrl} \
                {SUBSET_ARGS}
        else:
            print(f"[0秒跳过] {_job['label']} 的 TEST 评估已是最新。")

        # 基于刚保存的 subspace_encoded_cache_test.pt 1 秒补算 PreRoll M=4 (eta=0.8) 供同表参照
        _j_base = Path(DRIVE_RESULTS_DIR) / _job_ds / _job_tag / _job_stage
        for _s in SEED_LIST:
            for _pt_p in _j_base.glob(f'{SUBSET_GLOB}/seed-{int(_s):03d}/exploratory/subspace_encoded_cache_test.pt'):
                _meval_out = _pt_p.parent.parent / 'meval_fast_summary'
                if not (_meval_out / 'preroll_m_eval_tradeoff_test.json').is_file():
                    !cd {CODE_DIR} && {PYTHON} -u -m braindecoding.evaluation.common_mode_subtraction_evaluation \
                        --cache-pts {_pt_p} \
                        --output-dir {_meval_out} \
                        --split test \
                        --pos-etas 0.5 \
                        --global-etas 0.5 \
                        --preroll-etas 0.8 \
                        --m-eval-grid 4 \
                        --no-plot

    _test_rows = []
    # 1) 只读读取现有 Baseline seed-000 TEST 锚点（若云盘存在则读实时 JSON，否则使用已核验锚点参考）
    _base_loaded = False
    if BASELINE_REF_EXP_TAG and TRAIN_JOBS[0]['dataset'] == 'pallier2025':
        _b_root = Path(DRIVE_RESULTS_DIR) / 'pallier2025' / BASELINE_REF_EXP_TAG
        for _sub_cand in ('pos_causal_meval_audit', 'causal_fast_audit'):
            for _b_cms in (_b_root / _sub_cand / 'prefix_revision').glob(f'{SUBSET_GLOB}/seed-000/exploratory/common_mode_subtraction_test.json'):
                try:
                    _bd = json.loads(_b_cms.read_text(encoding='utf-8'))
                    for _ent in (_bd.get('sweep') or []):
                        _eta_b = float(_ent.get('eta', 0.0))
                        _mid_b = str(_ent.get('method_id', ''))
                        if abs(_eta_b) > 1e-9 and not _mid_b.startswith('causal_cms:'):
                            continue
                        _cr = _ent.get('clean_reference') or {}
                        _sa = _ent.get('shifted_aggregate') or {}
                        _c50 = float(_cr.get('vocab50_macro_top10', 0.0)) * 100.0
                        _s50 = float(_sa.get('mean_vocab50_macro_top10', _sa.get('shifted_mean_vocab50_macro_top10', 0.0))) * 100.0
                        _g50 = float(_sa.get('clean_minus_shifted_vocab50_macro_top10', (_c50 - _s50) / 100.0)) * 100.0
                        _test_rows.append({
                            'seed': 0,
                            'label': 'Base Raw(1x)' if abs(_eta_b) < 1e-9 else f'Base EMA({_eta_b:g},1x)',
                            'tier_prefix': 'Baseline',
                            'exp_tag': BASELINE_REF_EXP_TAG,
                            'm_eval': 0,
                            'c50': _c50,
                            's50': _s50,
                            'g50': _g50,
                            't1_50': float(_cr.get('vocab50_macro_top1', 0.0)) * 100.0,
                            'c_full': float(_cr.get('full_vocab_word_macro_top10', _cr.get('full_vocab_macro_top10', 0.0))) * 100.0,
                            'mic_full': float(_cr.get('full_vocab_micro_top10', 0.0)) * 100.0,
                        })
                        _base_loaded = True
                except Exception:
                    pass
            if _base_loaded:
                break
    if not _base_loaded and TRAIN_JOBS[0]['dataset'] == 'pallier2025' and 0 in SEED_LIST:
        _test_rows.extend([
            {'seed': 0, 'label': 'Base Raw(1x) [ref]', 'tier_prefix': 'Baseline', 'exp_tag': BASELINE_REF_EXP_TAG, 'm_eval': 0, 'c50': 57.41, 's50': 24.14, 'g50': 33.27, 't1_50': 13.91, 'c_full': 6.19, 'mic_full': 25.20},
            {'seed': 0, 'label': 'Base M=4(5x) [ref]', 'tier_prefix': 'Baseline', 'exp_tag': BASELINE_REF_EXP_TAG, 'm_eval': 4, 'c50': 55.46, 's50': 20.22, 'g50': 35.23, 't1_50': 13.50, 'c_full': 6.13, 'mic_full': 24.68},
        ])

    # 2) 读取 Tier A & Tier B 的 1x (Raw / Global-EMA) 与 PreRoll M=4 (5x)
    for _job in TRAIN_JOBS:
        _job_cms_rows = _collect_cms_rows_for_job(_job, 'test')
        _test_rows.extend(_job_cms_rows)
        _seen_dirs = {r['seed_dir'] for r in _job_cms_rows if r.get('seed_dir')}
        for _sdir_str in sorted(_seen_dirs):
            _tr_json = Path(_sdir_str) / 'meval_fast_summary' / 'preroll_m_eval_tradeoff_test.json'
            if _tr_json.is_file():
                try:
                    _tr_d = json.loads(_tr_json.read_text(encoding='utf-8'))
                    for _m in (_tr_d.get('methods') or []):
                        if str(_m.get('method_id', '')) == 'preroll_loo:M=4:eta=0.8':
                            _pfx = 'Tier A' if 'null_space' in _job['exp_tag'] else 'Tier B'
                            _test_rows.append({
                                'seed': int((_tr_d.get('seeds') or [0])[0]),
                                'label': f'{_pfx} M=4(5x,0.8)',
                                'tier_prefix': _pfx,
                                'exp_tag': _job['exp_tag'],
                                'm_eval': 4,
                                'c50': float(_m.get('clean_vocab50_macro_top10_mean', 0.0)) * 100.0,
                                's50': float(_m.get('shifted_vocab50_macro_top10_mean', 0.0)) * 100.0,
                                'g50': float(_m.get('gap_vocab50_macro_top10_mean', 0.0)) * 100.0,
                                't1_50': 0.0,
                                'c_full': float(_m.get('clean_full_vocab_word_macro_top10_mean', 0.0)) * 100.0,
                                'mic_full': float(_m.get('clean_full_vocab_micro_top10_mean', 0.0)) * 100.0,
                            })
                except Exception:
                    pass

    if _test_rows:
        _winner_1x_rows = [
            r for r in _test_rows
            if r.get('exp_tag') == _val_winner_exp_tag and int(r.get('m_eval', 0)) == 0
        ]
        _gate_passed = False
        _best_1x_winner = None
        if _winner_1x_rows:
            _passing = [
                r for r in _winner_1x_rows
                if r['s50'] < float(PROBE_GATE_MAX_SHIFTED_50W) and r['g50'] > float(PROBE_GATE_MIN_GAP_50W)
            ]
            if _passing:
                _gate_passed = True
                _best_1x_winner = max(_passing, key=lambda r: r['g50'])
            else:
                _best_1x_winner = max(_winner_1x_rows, key=lambda r: r['g50'])

        _tlines = [
            '=' * 86,
            f"Step 2 Hard-Gate Audit (TEST) | {TRAIN_JOBS[0]['dataset']} | VAL Winner: {_val_winner_prefix}",
            f"Gate Rule (1x, M_eval=0): 50w Shifted < {PROBE_GATE_MAX_SHIFTED_50W:.1f}% AND 50w Gap > +{PROBE_GATE_MIN_GAP_50W:.1f}%",
            '=' * 86,
            f"{'Method / Tier (Seed)':<24} | {'50wCln':>6} | {'50wShf':>6} | {'50wGap':>7} | {'50wTp1':>6} | {'FulMac':>6} | {'FulMic':>6}",
            '-' * 86,
        ]
        for _r in _test_rows:
            _nm = f"{_r['label']} s{_r['seed']}"
            _tp1_str = f"{_r['t1_50']:>5.2f}%" if _r.get('t1_50', 0.0) > 0 else '    - '
            _tlines.append(
                f"{_nm:<24} | {_r['c50']:>5.2f}% | {_r['s50']:>5.2f}% | {_r['g50']:>+6.2f}% | {_tp1_str} | {_r['c_full']:>5.2f}% | {_r['mic_full']:>5.2f}%"
            )
        _tlines.append('-' * 86)
        if _gate_passed and _best_1x_winner is not None:
            _tlines.append(
                f" [VERDICT = GO] 达标晋级！{_best_1x_winner['label']} 满足硬门槛 (Shf={_best_1x_winner['s50']:.2f}% < {PROBE_GATE_MAX_SHIFTED_50W:.1f}%, Gap={_best_1x_winner['g50']:+.2f}% > +{PROBE_GATE_MIN_GAP_50W:.1f}%)"
            )
            _tlines.append(
                f" -> 下一步扩跑 3 种子：RUN_TIER_AB_PROBE=False | CONFIG_FILE='{_val_winner_cfg}' | EXP_TAG='{_val_winner_exp_tag}' | SEED='0,1,2'"
            )
        elif _best_1x_winner is not None:
            _tlines.append(
                f" [VERDICT = STOP] 未过硬门槛 ({_best_1x_winner['label']}: Shf={_best_1x_winner['s50']:.2f}%, Gap={_best_1x_winner['g50']:+.2f}%) —— 立即停止训练！"
            )
            _tlines.append(
                " -> 结论：直接采用现有 3-seed 主基线 + Global-EMA(0.5) / PreRoll M=4 曲线进入论文撰写。"
            )
        _tlines.append('=' * 86)
        _board_test_txt = '\n'.join(_tlines)
        print('\n' + _board_test_txt)
        _cmp_test_out = Path(DRIVE_RESULTS_DIR) / TRAIN_JOBS[0]['dataset'] / f'probe_tier_ab_{ACTIVE_PROBE_SUFFIX}_comparison_test.md'
        _cmp_test_out.write_text('```text\n' + _board_test_txt + '\n```\n', encoding='utf-8')
        print(f'[已永久保存 TEST 硬门槛判决看板至云盘] {_cmp_test_out}')

_active_tags = {_j['exp_tag'] for _j in TRAIN_JOBS if _j.get('exp_tag')}
for seed_val in SEED_LIST:
    eval_pngs = sorted(
        Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(seed_val):03d}/figures/*_val*.png'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    shown = set()
    for png_path in eval_pngs:
        if _active_tags and not any(_t in png_path.parts for _t in _active_tags):
            continue
        _key = (tuple(_t for _t in _active_tags if _t in png_path.parts), png_path.name)
        if _key in shown:
            continue
        shown.add(_key)
        print(f'\n[图表预览 | seed-{int(seed_val):03d}] {png_path}')
        display(Image(filename=str(png_path)))

In [ ]:
# 最后一个单元格：同步落盘并按 AUTO_UNASSIGN_RUNTIME 决定是否自动释放 GPU
import os
import time
from google.colab import drive, runtime

# 1. 刷新系统层面的文件写缓冲
os.sync()

if AUTO_UNASSIGN_RUNTIME:
    # 2. 强制 Google Drive 将挂载目录中的所有缓存刷新并同步至云端
    drive.flush_and_unmount()
    # 3. 预留 10 秒缓冲时间，确保网络请求收尾
    print("数据已全部同步至 Google Drive，正在释放运行时以停止扣费...")
    time.sleep(10)
    # 4. 释放实例
    runtime.unassign()
else:
    print("AUTO_UNASSIGN_RUNTIME=False：已保留当前 Colab GPU 会话供交互查看。")